# Part 6 — Feature-Based Model (XGBoost)
**Appliance Energy Use Forecasting — 7PAM2033**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DolapoMichael/Time-Series-coding-Case-study-and-Report/blob/main/notebooks/06_feature_model.ipynb)

Trains an XGBoost regressor with time-series-aware hyperparameter tuning (`TimeSeriesSplit`, not a shuffled k-fold) and a feature-group ablation across time/lag/rolling/sensor feature groups. Takes a few minutes to run (125 total XGBoost fits during tuning) — much faster than Part 4's SARIMAX grid search.

**Important:** this model uses real historical lags and real future weather for the test rows (see Part 5), so it is evaluated as a **conditional** forecast, not a blind one like SARIMAX — not directly apples-to-apples without that caveat, which is carried through in the printed output below.

In [ ]:
!pip install -q xgboost scikit-learn

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor

RAW_CSV_URL = 'https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/energydata_complete.csv'
TARGET = 'Appliances'
DAILY_PERIOD = 24
TEST_DAYS = 14
TEST_STEPS = TEST_DAYS * DAILY_PERIOD
RANDOM_STATE = 0
LAGS = [1, 2, 3, 6, 12, 24, 48, 168]
ROLLING_WINDOWS = [3, 6, 12, 24, 168]
TIME_COLS = ['hour', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
LAG_COLS = [f'lag_{l}' for l in LAGS]

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
for d in [DATA_DIR, OUTPUT_DIR / 'forecasts', OUTPUT_DIR / 'metrics', OUTPUT_DIR / 'figures', OUTPUT_DIR / 'model_objects']:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

## Load or rebuild the feature table (self-contained)

In [ ]:
features_path = DATA_DIR / 'energydata_features.csv'

if features_path.exists():
    features = pd.read_csv(features_path, index_col=0, parse_dates=True)
else:
    print('No local feature table found — rebuilding from scratch...')
    hourly_path = DATA_DIR / 'energydata_hourly.csv'
    if hourly_path.exists():
        hourly = pd.read_csv(hourly_path, index_col=0, parse_dates=True)
    else:
        raw = pd.read_csv(RAW_CSV_URL)
        raw['date'] = pd.to_datetime(raw['date'])
        raw = raw.set_index('date').sort_index()
        energy_cols = ['Appliances', 'lights']
        sensor_cols = [c for c in raw.columns if c not in energy_cols + ['rv1', 'rv2']]
        hourly = pd.concat([raw[energy_cols].resample('h').sum(),
                             raw[sensor_cols].resample('h').mean()], axis=1)
        hourly.to_csv(hourly_path)

    def add_time_features(df):
        out = df.copy()
        out['hour'] = out.index.hour
        out['dayofweek'] = out.index.dayofweek
        out['is_weekend'] = (out['dayofweek'] >= 5).astype(int)
        out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24)
        out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24)
        out['dow_sin'] = np.sin(2 * np.pi * out['dayofweek'] / 7)
        out['dow_cos'] = np.cos(2 * np.pi * out['dayofweek'] / 7)
        return out

    out = add_time_features(hourly)
    for lag in LAGS:
        out[f'lag_{lag}'] = out[TARGET].shift(lag)
    for window in ROLLING_WINDOWS:
        shifted = out[TARGET].shift(1)
        out[f'roll_mean_{window}'] = shifted.rolling(window).mean()
        out[f'roll_std_{window}'] = shifted.rolling(window).std()
    features = out.dropna()
    features.to_csv(features_path)

print(f'Feature table shape: {features.shape}')

## Split and evaluation helper

In [ ]:
def evaluate_forecast(name, y_true, y_pred, y_train, seasonality=DAILY_PERIOD):
    y_pred = y_pred.reindex(y_true.index)
    valid = y_true.notna() & y_pred.notna()
    yt, yp = y_true.loc[valid], y_pred.loc[valid]
    y_train_f = y_train.astype(float)
    naive_err = np.abs(y_train_f.iloc[seasonality:].values - y_train_f.iloc[:-seasonality].values)
    scale = naive_err.mean()
    return {
        'model': name,
        'MAE': float(np.mean(np.abs(yt.values - yp.values))),
        'RMSE': float(np.sqrt(np.mean((yt.values - yp.values) ** 2))),
        'MASE': float(np.mean(np.abs(yt.values - yp.values)) / scale) if scale else float('nan'),
        'Bias': float(np.mean(yp.values - yt.values)),
        'n_points': int(valid.sum()),
    }

feature_cols = [c for c in features.columns if c != TARGET]
train = features.iloc[:-TEST_STEPS]
test = features.iloc[-TEST_STEPS:]
X_train, y_train = train[feature_cols], train[TARGET]
X_test, y_test = test[feature_cols], test[TARGET]
print(f'Train: {X_train.index.min()} to {X_train.index.max()} ({len(X_train)} obs)')
print(f'Test:  {X_test.index.min()} to {X_test.index.max()} ({len(X_test)} obs)')
print(f'Using {len(feature_cols)} features.')

## Feature-group ablation
Fixed hyperparameters (deliberately not tuned per subset) so the comparison isolates the effect of the feature groups themselves.

In [ ]:
rolling_cols = [c for c in features.columns if c.startswith('roll_')]
sensor_weather_cols = [c for c in features.columns if c not in TIME_COLS + LAG_COLS + rolling_cols + [TARGET]]

feature_sets = {
    'time_only': TIME_COLS,
    'time_plus_lag': TIME_COLS + LAG_COLS,
    'time_plus_lag_plus_rolling': TIME_COLS + LAG_COLS + rolling_cols,
    'all_features': TIME_COLS + LAG_COLS + rolling_cols + sensor_weather_cols,
}

ablation_rows = []
for name, cols in feature_sets.items():
    model = XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=RANDOM_STATE, n_jobs=-1, objective='reg:squarederror')
    model.fit(train[cols], y_train)
    pred = pd.Series(model.predict(test[cols]), index=test.index)
    m = evaluate_forecast(name, y_test, pred, y_train)
    ablation_rows.append(m)
    print(f"  {name:28s} ({len(cols):2d} features)  MASE={m['MASE']:.3f}  MAE={m['MAE']:.1f}")

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(OUTPUT_DIR / 'metrics' / 'xgboost_ablation.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(ablation_df['model'], ablation_df['MASE'], color='#1f5b8a')
ax.axhline(1.0, color='#c0392b', linestyle='--', linewidth=1, label='MASE=1 (seasonal-naive parity)')
ax.set_ylabel('MASE (lower is better)')
ax.set_title('Feature-group ablation — incremental value of each feature group')
ax.set_xticks(range(len(ablation_df)))
ax.set_xticklabels(ablation_df['model'], rotation=20, ha='right')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '06b_xgboost_ablation.png')
plt.show()

## Hyperparameter tuning
`TimeSeriesSplit` (5 folds), 25 random search iterations — each fold trains on the past and validates on the block immediately after, never validating on data earlier than its own training window (unlike a shuffled k-fold, which would leak future information into past training folds).

In [ ]:
param_distributions = {
    'n_estimators': [200, 400, 600, 800],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}

base_model = XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, objective='reg:squarederror')
tscv = TimeSeriesSplit(n_splits=5)

search = RandomizedSearchCV(
    base_model, param_distributions=param_distributions, n_iter=25,
    cv=tscv, scoring='neg_mean_absolute_error', random_state=RANDOM_STATE, n_jobs=-1,
)
search.fit(X_train, y_train)
print(f'Best CV MAE: {-search.best_score_:.2f} Wh')
print(f'Best params: {search.best_params_}')
best_model = search.best_estimator_

## Evaluate the tuned model on the full test period

In [ ]:
pred = pd.Series(best_model.predict(X_test), index=X_test.index, name='feature_model')
metrics = evaluate_forecast('xgboost_tuned', y_test, pred, y_train)
print(pd.Series(metrics))
print()
print('Note: this is a CONDITIONAL forecast (real historical lags + real future')
print('weather used for test rows) — not directly comparable to SARIMAX\'s blind')
print('forecast without that caveat. See Part 5 notes.')

## Feature importance

In [ ]:
importance = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
top = importance.head(20)

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top.index[::-1], top.values[::-1], color='#1f5b8a')
ax.set_xlabel('XGBoost feature importance (gain-based, normalised)')
ax.set_title('Top 20 features — tuned XGBoost model')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '06a_xgboost_feature_importance.png')
plt.show()

## Forecast vs. actual

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
features[TARGET].iloc[:-TEST_STEPS].tail(7 * DAILY_PERIOD).plot(ax=ax, label='Train (last 7 days)', color='#999999', linewidth=1)
y_test.plot(ax=ax, label='Actual (test)', color='black', linewidth=1.6)
pred.plot(ax=ax, label='XGBoost (conditional forecast)', color='#2e8a5b', linewidth=1.2, alpha=0.9)
ax.set_title('XGBoost vs. actual — full 336h test period (conditional forecast — see notes)')
ax.set_xlabel('Date'); ax.set_ylabel('Appliances (Wh / hour)')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '06c_xgboost_forecast_vs_actual.png')
plt.show()

## Save outputs

In [ ]:
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / 'metrics' / 'xgboost_metrics.csv', index=False)
pd.DataFrame({'actual': y_test, 'feature_model': pred}).to_csv(OUTPUT_DIR / 'forecasts' / 'xgboost_forecasts.csv')
best_model.save_model(str(OUTPUT_DIR / 'model_objects' / 'xgboost_final.json'))
print('Saved forecasts, metrics, ablation results, and the fitted model to outputs/')